# Ejercicio Módulo 5 - Dataset Swiss
**Inteligencia Artificial - CEIA - FIUBA**

**NICOLAS LASTRA**

Para aprender sobre regresión, vamos a utilizar un dataset clásico llamado Swiss, que proviene originalmente del lenguaje R. Este dataset contiene datos socioeconómicos de 47 provincias suizas a fines del siglo XIX. Cada fila representa una provincia, y las variables reflejan características demográficas y sociales relevantes para ese contexto histórico.

## Variables

- `Location`: Provincia donde se midieron los datos.
- `Fertility`: Tasa de fertilidad (número promedio de hijos por mujer)
- `Agriculture`:` Porcentaje de hombres ocupados en agricultura
- `Examination`: Porcentaje de hombres que completaron exámenes de educación superior
- `Education`: Nivel promedio de educación (escala arbitraria)
- `Catholic`: Porcentaje de población católica
- `Infant.Mortality`: Tasa de mortalidad infantil (por cada 1000 nacidos vivos)

## Que queremos predecir?

Vamos a utilizar este dataset para predecir la tasa de fertilidad en cada provincia mediante diferentes métodos de regresión.

--- 

Siguiendo el procedimiento típico de Machine Learning, vamos a leer los datos y separarlos en los datasets de entrenamiento y testeo utilizando Scikit-Learn...

In [74]:
import pandas as pd

df = pd.read_csv("swiss.csv")

df.head()

,Location,Fertility,Agriculture,Examination,Education,Catholic,Infant.Mortality
0,Courtelary,80.2,17.0,15,12,9.96,22.2
1,Delemont,83.1,45.1,6,9,84.84,22.2
2,Franches-Mnt,92.5,39.7,5,5,93.40,20.2
3,Moutier,85.8,36.5,12,7,33.77,20.3
4,Neuveville,76.9,43.5,17,15,5.16,20.6


In [75]:
print(f"Tenemos {df.shape[0]} observaciones")

Tenemos 47 observaciones


Obtenemos la variable objetivo (`Fertility`) y, por otro lado, los atributos (quitamos `Location` ya que no es un atributo numérico relevante para la regresión)

In [76]:
X = df.drop(["Fertility", "Location"], axis=1)
y = df["Fertility"]

Dado que tenemos pocas observaciones, vamos a separar el dataset en un 50% para entrenamiento y 50% para testeo:

In [77]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)


## Regresión lineal múltiple

Arranquemos la primera parte del ejercicio. Para eso, vamos a entrenar un modelo de regresión lineal múltiple usando todos los atributos. Para ello debes:

1. Escalar los atributos usando `StandardScaler`
2. Entrenar el modelo usando el dataset de entrenamiento.
3. Obtener las predicciones sobre el dataset de testeo.
4. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.

In [78]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

##### COMPLETAR AQUI LO PEDIDO

#Estandarizamos
estandarizador = StandardScaler() #Creamos el objeto estandarizador
normalizador = MinMaxScaler() #Creamos el objeto normalizador

X_train_estandarizada = estandarizador.fit_transform(X_train) #Entrenamos el escalador con los datos y estandarizamos el set de entrenamiento.
X_train_estandarizada_normalizada = normalizador.fit_transform(X_train_estandarizada)

X_test_estandarizada = estandarizador.transform(X_test) #Sin entrenar solo estandarizamos los datos del set de testeo.
X_test_estandarizada_normalizada = normalizador.transform(X_test_estandarizada) #Sin entrenar solo estandarizamos los datos del set de testeo.

#Entrenamos el modelo.
model_linear_regression = LinearRegression()
model_linear_regression.fit(X_train_estandarizada_normalizada, y_train)

#Predecimos con el dataset de testeo.
y_test_predicted = model_linear_regression.predict(X_test_estandarizada_normalizada)

#Obtener y mostrar metricas
r2 = r2_score(y_test, y_test_predicted) #Coeficiente de determinación
mae = mean_absolute_error(y_test, y_test_predicted) #Error absoluto medio
mse = mean_squared_error(y_test, y_test_predicted) #Error cuadratico medio

print("R-cuadrado en test:", round(r2, 2))
print("Error absoluto medio:", round(mae, 2))
print("Error cuadrático medio:", round(mse, 2))



R-cuadrado en test: 0.57
Error absoluto medio: 6.09
Error cuadrático medio: 64.46


## Modelo con regularización

Para mejorar nuestro modelo, vamos a explorar técnicas de regresión lineal con regularización, que nos permiten controlar el sobreajuste y seleccionar variables relevantes automáticamente.

Existen dos variantes muy populares:

- Una penaliza la suma de los cuadrados de los coeficientes (regularización L2).
- La otra penaliza la suma del valor absoluto de los coeficientes (regularización L1).

Ambas ayudan a mejorar la generalización, pero una de ellas además puede eliminar variables (coeficientes exactamente cero), lo que ayuda a identificar qué atributos son realmente importantes.

Tu tarea:

1. Elegí correctamente cuál de los dos métodos de regularización usar para este problema. 
    - Pista: Queremos que el modelo sea capaz de hacer una selección automática de variables, dejando fuera aquellas que no aportan.
2. Implementá un pipeline que incluya escalado y el modelo elegido.
3. Buscá automáticamente el mejor valor del hiperparámetro de regularización (alpha) usando validación cruzada usando 3-folds.
4. Entrená el modelo con los datos de entrenamiento y obtené las predicciones para el set de testeo.
5. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.
6. Imprimí los coeficientes resultantes e identificá qué variables fueron eliminadas (coeficiente = 0).

In [79]:
import numpy as np

from sklearn.linear_model import LassoCV, RidgeCV

Paremos un momento para entender qué hacen LassoCV y RidgeCV antes de continuar con la resolución:

> Tanto `LassoCV` como `RidgeCV` son implementaciones de regresión lineal con regularización que incluyen la búsqueda automática del mejor hiperparámetro alpha mediante validación cruzada.
>
> Ambos métodos prueban distintos valores de alpha y eligen el que minimiza el error del modelo, facilitando el proceso de ajuste sin necesidad de una búsqueda manual.
>
> Internamente, utilizan la métrica del error cuadrático medio (MSE) para evaluar el rendimiento del modelo en cada fold de la validación cruzada.
>
> Por ejemplo, si llamás a RidgeCV(alphas=alphas, cv=5), se hará una validación cruzada de 5 folds utilizando los valores de alpha que vos le pases, y se seleccionará el que obtenga el menor MSE promedio.
> 
> Una vez elegido el mejor alpha, el modelo final se entrena con todos los datos de entrenamiento usando ese valor.

¡Listo! Con todo lo que vimos hasta ahora, ya estás en condiciones de resolver esta parte y completar los 6 puntos propuestos

In [80]:
alphas = np.logspace(-4, 1, 500)

##### COMPLETAR AQUI LO PEDIDO

#1 - El metodo seleccionado es Regresion de Lasso.

#2 y 3 - Se crea el objeto pipeline, con la estandarizacion y normalizacion de los datos, la regresion de lasso con VC 3-folds y ajustando al mejor alpha.

pipeline = Pipeline(steps=[
    ('estandarizador', StandardScaler()),                        # Estandarizada los datos.
    ('normalizador', MinMaxScaler()),                            # Normaliza los  datos.
    ('regresor', LassoCV(alphas=alphas, cv=3, random_state=42))  # Aplica la regresión Lasso con validación cruzada 3-folds y los alphas propuestos.
])

#4 - Entrenamiento del pipeline con los datos de entrenamiento, ya estandarizados anteriormente y se obtienen las predicciones.
pipeline.fit(X_train, y_train)
y_test_predicted_lasso = pipeline.predict(X_test)

5# - Calculo de las metricas correspondientes

r2 = r2_score(y_test, y_test_predicted_lasso) #Coeficiente de determinación
mae = mean_absolute_error(y_test, y_test_predicted_lasso) #Error absoluto medio
mse = mean_squared_error(y_test, y_test_predicted_lasso) #Error cuadratico medio

print("R-cuadrado en test:", round(r2, 2))
print("Error absoluto medio:", round(mae, 2))
print("Error cuadrático medio:", round(mse, 2),"\n")

6# - Se muestran los coeficientes de la regresion y cuales columnas fueron eliminadas.

for index, coeficiente in enumerate(pipeline.named_steps['regresor'].coef_):
    print(f'El coeficiente {index} tiene un valor: {coeficiente}')
print("\n")

nombres_columnas = X.columns
coeficientes = pipeline.named_steps['regresor'].coef_

for nombre, coef in zip(nombres_columnas, coeficientes):
    estado = "ELIMINADA" if coef == 0.0 else "ACTIVA"
    print(f"{nombre}: {coef:.2f} ({estado})")

R-cuadrado en test: 0.56
Error absoluto medio: 6.07
Error cuadrático medio: 66.06 

El coeficiente 0 tiene un valor: -0.0
El coeficiente 1 tiene un valor: -0.0
El coeficiente 2 tiene un valor: -11.352690902926284
El coeficiente 3 tiene un valor: 9.09041895706935
El coeficiente 4 tiene un valor: 10.592630932337753


Agriculture: -0.00 (ELIMINADA)
Examination: -0.00 (ELIMINADA)
Education: -11.35 (ACTIVA)
Catholic: 9.09 (ACTIVA)
Infant.Mortality: 10.59 (ACTIVA)


## Comparación de modelos y conclusiones

Completá la siguiente tabla con las métricas obtenidas para cada uno de los modelos que entrenaste:

| Modelo                        | MAE | MSE | $R^2$ |
| ----------------------------- | --- | --- | ----- |
| Regresión Lineal              |  6.09   |  64.46  |  0.57  |
| Regresión Lasso               |  6.07   |  66.06  |  0.56  |


> ⚠️ Asegurate de cambiar el nombre del modelo `Modelo Regularizado (L1 o L2)` según el modelo que usaste (Lasso o Ridge).

### Justificación

**¿Cuál de los modelos te parece que tuvo un mejor desempeño general?**

Tené en cuenta las tres métricas al responder, y también pensá en la complejidad del modelo (por ejemplo, si eliminó variables innecesarias).

Escribí tu respuesta a continuación:

ESCRIBA AQUI SU RESPUESTA

Luego de analizar el dataset a traves de la regresion lineal estándar y la regresion Lasso, se puede observar que las metricas son muy similares 
no mejorando en este caso la performance a nivel prediccion, pero si se obtuvo una mejora a nivel tamaño de datos (complejidad), ya que se logro precindir de dos columnas "Agriculture" y "Examination", por lo que el mejor modelo fue Lasso.